- [x] Data Loading
- [x] Cleaning registration required fields
- [x] Cleaning session_ping required fields
- [ ] Checking types of each fields
- [ ] Checking valid values for `state`, `outcome`,...

# Imports

In [61]:
import sys
import os
import json
import pandas as pd
import numpy as np
from rich import print

from typing import List, Dict

In [2]:
# Import google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data loading

Uvesti da se fajl ili upload ili da treba da izmene putanju

In [94]:
# Path to Nordeus folder
folder_path = '/content/drive/MyDrive/Nordeus Data Engineering Challenge'

In [95]:
file_name = 'events.jsonl'
full_file_path = os.path.join(folder_path, file_name)

events_data = []
try:
    with open(full_file_path, 'r') as f:
        for line in f:
            events_data.append(json.loads(line))
    print(f"Successfully loaded {len(events_data)} events from '{file_name}'.")
    events_df = pd.DataFrame(events_data)
    print("Data loaded into 'events_df' DataFrame.")
except FileNotFoundError:
    print(f"Error: The file '{file_name}' was not found at '{full_file_path}'. Please check the file name and path.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON from '{file_name}': {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Successfully loaded 5000 events from 'events.jsonl'.

Data loaded into 'events_df' DataFrame.

In [96]:
file_name = 'maps.jsonl'
full_file_path = os.path.join(folder_path, file_name)

maps_data = []
try:
    with open(full_file_path, 'r') as f:
        for line in f:
            maps_data.append(json.loads(line))
    print(f"Successfully loaded {len(maps_data)} events from '{file_name}'.")
    maps_df = pd.DataFrame(maps_data)
    print("Data loaded into 'maps_df' DataFrame.")
except FileNotFoundError:
    print(f"Error: The file '{file_name}' was not found at '{full_file_path}'. Please check the file name and path.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON from '{file_name}': {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully loaded 5 events from 'maps.jsonl'.

Data loaded into 'maps_df' DataFrame.

# Data cleaning

List `events_data` contains event data as list of JSONs

---

`events_df` contains data in pandas dataframe

List `maps_data` contains maps data as list of JSONs

---

`maps_df` contains data in pandas dataframe



In [97]:
events_df['event_type'].unique()

array(['registration', 'session_ping', 'match_start', 'match_finish'],
      dtype=object)

## Checking missing fields in `event_data` field

In [98]:
# for given `event_type` returns id's that do not have required fields
missing_event_type_dict: dict[str, list[int]] = dict()

### Checking `registration` `event_type`


We have all regular event_type names

In [99]:
events_registration_df = events_df[events_df['event_type']=='registration'].copy()

In [100]:
events_registration_df.head()

,id,timestamp,event_type,user_id,event_data
0,2790,1775171101,registration,5129c6fc-f112-4675-ad99-af28f058a59c,"{'country': 'DEU', 'device_os': 'iOS', 'userna..."
8,799,1775176298,registration,57dc71fe-00cb-4b2f-8a74-4aa1fb127844,"{'country': 'BIH', 'device_os': 'Android', 'us..."
9,636,1775179205,registration,f79f5ff5-1bf5-4947-89e5-b688afe4030d,"{'country': 'SRB', 'device_os': 'iOS', 'userna..."
10,636,1775179205,registration,f79f5ff5-1bf5-4947-89e5-b688afe4030d,"{'country': 'SRB', 'device_os': 'iOS', 'userna..."
16,185,1775179688,registration,bb5f9d21-16d4-4587-938d-53afacf47cb1,"{'country': 'DEU', 'device_os': 'iOS', 'userna..."


In [101]:
required_fields = ['country', 'device_os', 'username'] # Assuming 'device' refers to 'device_os'

'''
  Args:
    data_dict
'''
def check_required_fields(data_dict: Dict,
                          fields: List) -> bool:
    if not isinstance(data_dict, dict):
        return False # Not a dictionary, so fields can't be present

    for field in fields:
      if field not in data_dict:
        return False
    return True

# Apply the check to the 'event_data' column
events_registration_df['has_required_fields'] = events_registration_df['event_data'].apply(lambda x: check_required_fields(x, required_fields))

# Count how many rows have all required fields
num_rows_with_all_fields = events_registration_df['has_required_fields'].sum()

print(f"Number of registration events with all required fields ({', '.join(required_fields)}): {num_rows_with_all_fields}")
print(f"Total registration events: {len(events_registration_df)}")

# Display some rows to show the new column
print("\nRegistration events with new 'has_required_fields' column (first 5 rows):")
print(events_registration_df[['event_data', 'has_required_fields']].head())

Number of registration events with all required fields (country, device_os, username): 51

Total registration events: 53

Registration events with new 'has_required_fields' column (first 5 rows):

event_data  has_required_fields
0   {'country': 'DEU', 'device_os': 'iOS', 'userna...                 True
8   {'country': 'BIH', 'device_os': 'Android', 'us...                 True
9   {'country': 'SRB', 'device_os': 'iOS', 'userna...                 True
10  {'country': 'SRB', 'device_os': 'iOS', 'userna...                 True
16  {'country': 'DEU', 'device_os': 'iOS', 'userna...                 True

In [102]:
# Find rows where 'has_required_fields' is False
false_required_fields_df = events_registration_df[events_registration_df['has_required_fields'] == False]

# Extract the 'id' column and convert it to a list
ids_with_missing_fields = false_required_fields_df['id'].tolist()

print(f"Number of events with missing required registration fields: {len(ids_with_missing_fields)}")
print("IDs of events with missing required fields:")
print(ids_with_missing_fields)

missing_event_type_dict['registration'] = ids_with_missing_fields

Number of events with missing required registration fields: 2

IDs of events with missing required fields:

[1881, 1384]

### Checking `session_ping` `event_type`

In [103]:
events_session_df = events_df[events_df['event_type']=='session_ping'].copy()

In [104]:
events_session_df.head()

,id,timestamp,event_type,user_id,event_data
1,2791,1775174844,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'started', 'device_os': 'Android'}"
2,2792,1775174964,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'in_progress', 'device_os': 'Android'}"
3,2793,1775175084,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'in_progress', 'device_os': 'Android'}"
4,2794,1775175204,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'in_progress', 'device_os': 'Android'}"
5,2795,1775175324,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'in_progress', 'device_os': 'Android'}"


In [105]:
required_fields = ['state', 'device_os']

# Apply the check to the 'event_data' column
events_session_df['has_required_fields'] = events_session_df['event_data'].apply(lambda x: check_required_fields(x, required_fields))

# Count how many rows have all required fields
num_rows_with_all_fields = events_session_df['has_required_fields'].sum()

print(f"Number of registration events with all required fields ({', '.join(required_fields)}): {num_rows_with_all_fields}")
print(f"Total registration events: {len(events_session_df)}")

Number of registration events with all required fields (state, device_os): 3776

Total registration events: 3954

In [106]:
# Find rows where 'has_required_fields' is False
false_required_fields_df = events_session_df[events_session_df['has_required_fields'] == False]

# Extract the 'id' column and convert it to a list
ids_with_missing_fields = false_required_fields_df['id'].tolist()

print(f"Number of events with missing required session_ping fields: {len(ids_with_missing_fields)}")
print("IDs of events with missing required fields:")
print(ids_with_missing_fields)

missing_event_type_dict['session_ping'] = ids_with_missing_fields

Number of events with missing required session_ping fields: 178

IDs of events with missing required fields:

[
    644,
    3400,
    1892,
    1893,
    3811,
    3814,
    526,
    3405,
    804,
    3408,
    532,
    2806,
    2809,
    4754,
    66,
    2032,
    2833,
    2837,
    812,
    663,
    664,
    668,
    389,
    2253,
    3833,
    3838,
    942,
    221,
    2265,
    4110,
    954,
    2165,
    2167,
    2171,
    673,
    427,
    2406,
    4220,
    243,
    3856,
    4116,
    4368,
    88,
    1948,
    551,
    4786,
    1953,
    438,
    440,
    2189,
    1428,
    2863,
    4243,
    560,
    4382,
    4798,
    4800,
    4802,
    1197,
    4133,
    573,
    3719,
    3720,
    3314,
    583,
    4264,
    2518,
    4901,
    970,
    4490,
    4394,
    3985,
    3893,
    3325,
    1108,
    3725,
    3,
    3002,
    3160,
    3006,
    1449,
    4411,
    457,
    3017,
    3471,
    979,
    2941,
    4691,
    4692,
    3026,
    2695,
    1325,
    5075,
    2332,
    4606,
    2535,
    1118,
    859,
    2077,
    986,
    991,
    21,
    23,
    4515,
    3475,
    3477,
    724,
    4161,
    3992,
    4939,
    1668,
    2708,
    2206,
    4532,
    272,
    1122,
    4415,
    4420,
    2548,
    3088,
    2949,
    1540,
    5106,
    1545,
    477,
    998,
    1124,
    112,
    113,
    2438,
    2443,
    3339,
    3341,
    2954,
    883,
    3624,
    2560,
    1475,
    4553,
    3348,
    1988,
    3047,
    1006,
    1819,
    1824,
    3756,
    1990,
    4713,
    1032,
    738,
    740,
    1037,
    2732,
    1253,
    3239,
    1831,
    2561,
    3584,
    3766,
    4429,
    295,
    2568,
    2101,
    2632,
    327,
    611,
    2585,
    1727,
    2891,
    4437,
    1492,
    1013,
    2894,
    45,
    2224,
    3490,
    4577,
    1863
]

### Checking `match_start` `event_type`

In [107]:
events_match_start_df = events_df[events_df['event_type']=='match_start'].copy()

In [108]:
events_match_start_df.head()

,id,timestamp,event_type,user_id,event_data
82,5369,1775207066,match_start,031ddc37-6878-4af6-b043-09a5ad23b653,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...
171,5896,1775234699,match_start,5129c6fc-f112-4675-ad99-af28f058a59c,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...
172,5895,1775234699,match_start,55046966-a746-4654-9d96-8464f7a8312b,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...
176,5610,1775234810,match_start,5129c6fc-f112-4675-ad99-af28f058a59c,{'map_id': 'db6f0747-d360-4427-be0c-82b78e51f7...
296,5656,1775252735,match_start,f79f5ff5-1bf5-4947-89e5-b688afe4030d,{'map_id': '42093495-255a-47b5-bdcd-347451b9f3...


In [109]:
required_fields = ['map_id', 'opponent_id']

# Apply the check to the 'event_data' column
events_match_start_df['has_required_fields'] = events_match_start_df['event_data'].apply(lambda x: check_required_fields(x, required_fields))

# Count how many rows have all required fields
num_rows_with_all_fields = events_match_start_df['has_required_fields'].sum()

print(f"Number of registration events with all required fields ({', '.join(required_fields)}): {num_rows_with_all_fields}")
print(f"Total registration events: {len(events_match_start_df)}")

Number of registration events with all required fields (map_id, opponent_id): 474

Total registration events: 498

In [110]:
# Find rows where 'has_required_fields' is False
false_required_fields_df = events_match_start_df[events_match_start_df['has_required_fields'] == False]

# Extract the 'id' column and convert it to a list
ids_with_missing_fields = false_required_fields_df['id'].tolist()

print(f"Number of events with missing required match_start fields: {len(ids_with_missing_fields)}")
print("IDs of events with missing required fields:")
print(ids_with_missing_fields)

missing_event_type_dict['match_start'] = ids_with_missing_fields

Number of events with missing required match_start fields: 24

IDs of events with missing required fields:

[
    5251,
    5818,
    5538,
    5397,
    5613,
    5339,
    5785,
    6190,
    5511,
    5884,
    6344,
    5308,
    6151,
    5375,
    5723,
    6398,
    6398,
    5167,
    5208,
    6166,
    6134,
    5955,
    5557,
    5218
]

### Checking `match_finish` `event_type`

In [111]:
events_match_finish_df = events_df[events_df['event_type']=='match_finish'].copy()

In [112]:
events_match_finish_df.head()

,id,timestamp,event_type,user_id,event_data
89,5370,1775207423,match_finish,031ddc37-6878-4af6-b043-09a5ad23b653,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...
90,5371,1775207423,match_finish,ae034bd0-3925-4259-b641-44d013a77d18,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...
178,5897,1775234834,match_finish,55046966-a746-4654-9d96-8464f7a8312b,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...
179,5898,1775234834,match_finish,5129c6fc-f112-4675-ad99-af28f058a59c,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...
181,5612,1775234933,match_finish,55046966-a746-4654-9d96-8464f7a8312b,{'map_id': 'db6f0747-d360-4427-be0c-82b78e51f7...


In [113]:
required_fields = ['map_id', 'opponent_id', 'outcome']

# Apply the check to the 'event_data' column
events_match_finish_df['has_required_fields'] = events_match_finish_df['event_data'].apply(lambda x: check_required_fields(x, required_fields))

# Count how many rows have all required fields
num_rows_with_all_fields = events_match_finish_df['has_required_fields'].sum()

print(f"Number of registration events with all required fields ({', '.join(required_fields)}): {num_rows_with_all_fields}")
print(f"Total registration events: {len(events_match_finish_df)}")

Number of registration events with all required fields (map_id, opponent_id, outcome): 468

Total registration events: 495

In [114]:
# Find rows where 'has_required_fields' is False
false_required_fields_df = events_match_finish_df[events_match_finish_df['has_required_fields'] == False]

# Extract the 'id' column and convert it to a list
ids_with_missing_fields = false_required_fields_df['id'].tolist()

print(f"Number of events with missing required match_finish fields: {len(ids_with_missing_fields)}")
print("IDs of events with missing required fields:")
print(ids_with_missing_fields)

missing_event_type_dict['match_finish'] = ids_with_missing_fields

Number of events with missing required match_finish fields: 27

IDs of events with missing required fields:

[
    5195,
    5388,
    6146,
    6383,
    6069,
    6192,
    6266,
    6154,
    5503,
    6085,
    5458,
    6055,
    5556,
    5315,
    5850,
    6066,
    5360,
    6374,
    5188,
    6270,
    5407,
    5148,
    6160,
    6357,
    5558,
    6058,
    5744
]

### Removing uncomplete `registration` and `session_ping` fields

In [120]:
registration_ids = missing_event_type_dict['registration']
session_ping_ids = missing_event_type_dict['session_ping']

ids_to_remove = registration_ids + session_ping_ids

events_df = events_df[~events_df['id'].isin(ids_to_remove)]

print(f"Removed {len(ids_to_remove)} rows from events_df.")
print(f"New events_df shape: {events_df.shape}")

Removed 182 rows from events_df.

New events_df shape: (4820, 5)

In [122]:
match_start_ids = missing_event_type_dict['match_start']
match_finish_ids = missing_event_type_dict['match_finish']

Prvo gledamo match_start podatke

In [177]:
for id in match_start_ids:
  p=events_df[events_df['id']==id]
  print(p['event_data'].iloc[0])

{'map_id': '42093495-255a-47b5-bdcd-347451b9f3bf'}

{'opponent_id': '84304216-13ec-4d6c-ad86-ef964b7ecd30'}

{'opponent_id': '4c800e05-5cfd-4884-a022-816d0ed5eb21'}

{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c059f'}

{'map_id': 'db6f0747-d360-4427-be0c-82b78e51f707'}

...

{'opponent_id': '727ea47d-d720-419c-a82b-3c63ad149888'}

{'opponent_id': '6d2b2743-af93-4df3-8f5a-2192032f40c9'}

{'opponent_id': 'afd79b8a-24db-4c4f-96e9-d33d2c57639d'}

{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c059f'}

{'map_id': 'db6f0747-d360-4427-be0c-82b78e51f707'}

Ne mozemo samo sa jednim od ova 2 podatka da nadjemo tacno ko nam je bio protivnik

In [178]:
events_df = events_df[~events_df['id'].isin(match_start_ids)]

Sada gledamo match_finish podatke. Ako nadjemo mec gde ima map_id ako i opponent_id, a fali samo outcome onda bi to moglo da sracuna

In [183]:
for id in match_finish_ids:
  p=events_df[events_df['id']==id]
  event_data = p['event_data'].iloc[0]
  if 'map_id' in event_data and 'opponent_id' in event_data:
    print(event_data)

{'map_id': '42093495-255a-47b5-bdcd-347451b9f3bf', 'opponent_id': 'e6fcde77-13b9-41ef-8eba-15657f54e0cb'}

{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c059f', 'opponent_id': 'a0289a40-bcc5-43d4-bebd-f7f5c94bf660'}

{'map_id': '55d20063-c89a-42c4-932d-e1af17936576', 'opponent_id': 'c1014f2f-e8bf-48d6-bf24-dbbefc8124a0'}

{'map_id': '42093495-255a-47b5-bdcd-347451b9f3bf', 'opponent_id': '3e6d2e88-2111-4c82-bca0-1a6d84fa39f7'}

{'map_id': '89e9ef59-d7df-459a-b558-52b00d5b8f91', 'opponent_id': 'ca340f13-cd88-4d12-b2fb-2d0e81e2640e'}

{'map_id': '55d20063-c89a-42c4-932d-e1af17936576', 'opponent_id': '0d6d4b4d-940d-4f1e-bf74-7af490b1c902'}

{'map_id': '89e9ef59-d7df-459a-b558-52b00d5b8f91', 'opponent_id': '4a501024-6855-45d3-b89c-6534d961c8c7'}

{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c059f', 'opponent_id': 'ca340f13-cd88-4d12-b2fb-2d0e81e2640e'}

{'map_id': '42093495-255a-47b5-bdcd-347451b9f3bf', 'opponent_id': '8e9525f8-1ffd-4aae-b7f7-d55bcbf2b089'}

{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c059f', 'opponent_id': '4c800e05-5cfd-4884-a022-816d0ed5eb21'}

proveri da uzmes podatke koji imaju outcome zato sto outcome postoji samo u match_finish, vidi da li samo pomocu opponent_id moze da se nadje mozda njegov protivnik pa da se gleda timestamp da je isti, i onda bi upario zajednicki map_id

In [185]:
specific_user_id = input("Please enter the specific user_id: ")
specific_map_id = input("Please enter the specific map_id: ")

filtered_events = events_df[
    (events_df['event_type'] == 'match_start') &
    (events_df['user_id'] == specific_user_id) &
    (events_df['event_data'].apply(lambda x: x.get('map_id') == specific_map_id))
]

print(f"Found {len(filtered_events)} events matching the criteria.")
if not filtered_events.empty:
    print("Matching events:")
    print(filtered_events)

Please enter the specific user_id: 4c800e05-5cfd-4884-a022-816d0ed5eb21
Please enter the specific map_id: fa508820-49dd-472a-b3bb-88931d7c059f


Found 2 events matching the criteria.

Matching events:

id   timestamp   event_type                               user_id  \
730   5539  1775300674  match_start  4c800e05-5cfd-4884-a022-816d0ed5eb21   
1864  5602  1775377921  match_start  4c800e05-5cfd-4884-a022-816d0ed5eb21   

                                             event_data  
730   {'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...  
1864  {'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...